# Paired RNA–protein BMI analysis

Bivariate Student-t models for matched RNA and protein signature scores, with numerical diagnostics, sensitivity fits and a paired within-plex bootstrap. Configure the input archives and output directory in the first code cell. Run cells in order in a CPU Colab runtime. Configure MODE and SHARD_ID: run workers 0–3, then aggregate the four completed shards. The saved default is aggregate mode. Original package pins, scientific settings, checkpoint logic and integrated diagnostic reports are retained.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, zipfile, tempfile, importlib.metadata
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/new_dysfunction')
INPUT_ZIP = PROJECT_ROOT / 'multiomics_inputs/PDAC_main_cohort_multiomics_inputs.zip'
SIGNATURE_INPUT_ZIP = PROJECT_ROOT / 'multiomics_inputs/PDAC_original_signature_inputs.zip'
OUTPUT_BASE = PROJECT_ROOT / 'multiomics_extensions/rna_protein_original_signatures_v2'
WORK = Path('/content/pdac_original_signatures_v2')
WORK.mkdir(parents=True, exist_ok=True)

for path in [INPUT_ZIP, SIGNATURE_INPUT_ZIP]:
    if not path.is_file():
        nearby = sorted(p.name for p in path.parent.iterdir()) if path.parent.is_dir() else []
        raise FileNotFoundError(f'Missing: {path}. Mount the Google account containing new_dysfunction. '
                                f'Files in that folder: {nearby}')
    print('Found:', path, f'({path.stat().st_size:,} bytes)')

with zipfile.ZipFile(INPUT_ZIP) as z:
    for suffix in ['prepared/matrices/rna_tumor_main_cohort.npz',
                   'prepared/matrices/protein_tumor_main_cohort.npz',
                   'prepared/model_covariates.csv', 'prepared/main_cohort_127.csv',
                   'prepared/tumor_plex_map.csv', 'prepared/preparation_contract.json',
                   'prepared/uploaded_export_membership_in_main_cohort.csv']:
        found = [name for name in z.namelist() if name.endswith('/' + suffix)]
        if len(found) != 1:
            raise ValueError(f'Expected exactly one {suffix}; found {found}')

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(dir=OUTPUT_BASE, prefix='path_check_', delete=False) as f:
    probe = Path(f.name)
    f.write(b'Drive write check')
try:
    assert probe.read_bytes() == b'Drive write check'
finally:
    probe.unlink(missing_ok=True)
print('Output folder checked:', OUTPUT_BASE)


In [ ]:
critical = ['numpy', 'scipy', 'pandas', 'pymc', 'arviz', 'pytensor', 'xarray', 'h5py']
loaded_before = {p: getattr(sys.modules[p], '__version__', None) for p in critical if p in sys.modules}
requirements = [
    'pymc==5.25.1', 'arviz==0.22.0', 'numpy>=2.0,<2.3',
    'scipy>=1.11,<1.16', 'pandas>=2.2,<3', 'xarray>=2025.1,<2026',
    'matplotlib>=3.8,<4', 'h5netcdf>=1.3,<2', 'h5py>=3.10,<4',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--disable-pip-version-check', '--upgrade-strategy', 'only-if-needed',
                *requirements], check=True)
changed = [p for p, v in loaded_before.items() if v and v != importlib.metadata.version(p)]
if changed:
    raise RuntimeError('Packages updated after import: ' + ', '.join(changed) +
                       '. Choose Runtime → Restart session, then Run all again.')
print('Colab Python:', sys.version.split()[0], '| sampler: PyMC NUTS')


In [ ]:
os.environ['PYTENSOR_FLAGS'] = 'blas__ldflags='   # use NumPy's BLAS; avoid MKL clashing with OpenBLAS
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
import time, traceback, warnings, types
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
from scipy.linalg import helmert
from scipy.stats import norm
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)

LAYERS = ['RNA', 'Protein']
ANALYSIS_VERSION = 'original_signatures_v2.0.0'
PROFILE = 'publication'                 # Change to 'smoke' only for troubleshooting.
SAMPLER = 'pymc'
RUN_INFLUENCE_REFITS = True
RUN_COMPOSITION_SENSITIVITY = True
STOP_ON_FIT_ERROR = True
FORCE_REFIT_FAILED = False
RUN_VARIANTS = ['primary', 'fixed_plex', 'export_subset_96', 'wes_purity_96',
                'stage_subset_base', 'stage_adjusted', 'BMI_at_least_18_5']
if PROFILE not in ['publication', 'smoke']:
    raise ValueError('PROFILE must be publication or smoke')
profile = ({'CHAINS':4, 'TUNE':2000, 'DRAWS':4000, 'PPC_DRAWS':1000,
            'BOOTSTRAP_REPLICATES':2000} if PROFILE == 'publication' else
           {'CHAINS':2, 'TUNE':150, 'DRAWS':150, 'PPC_DRAWS':150,
            'BOOTSTRAP_REPLICATES':100})
if PROFILE == 'smoke':
    RUN_VARIANTS = ['primary', 'fixed_plex']
    RUN_INFLUENCE_REFITS = RUN_COMPOSITION_SENSITIVITY = False
CONFIG = dict(PROFILE=PROFILE, INPUT_ZIP=str(INPUT_ZIP), OUTPUT_BASE=str(OUTPUT_BASE),
    SEED=20260920, SAMPLER=SAMPLER, CORES=2,
    TARGET_ACCEPT=.99, MAX_TREEDEPTH=12, BMI_PRIOR_SD=.5, COVARIATE_PRIOR_SD=.5,
    GROUP_SD_PRIOR=.3, FIXED_GROUP_PRIOR_SD=2., RETRY_ONCE=PROFILE=='publication',
    FORCE_REFIT_FAILED=FORCE_REFIT_FAILED, STOP_ON_FIT_ERROR=STOP_ON_FIT_ERROR,
    RUN_INFLUENCE_REFITS=RUN_INFLUENCE_REFITS,
    RUN_COMPOSITION_SENSITIVITY=RUN_COMPOSITION_SENSITIVITY, RUN_VARIANTS=RUN_VARIANTS, **profile)
print('Profile:', PROFILE)
print({p:importlib.metadata.version(p) for p in ['pymc','arviz','numpy','scipy','pandas','pytensor']})

# Execution settings only: identical scientific settings in every session.
MODE = 'aggregate'  # 'worker' or 'aggregate'
N_SHARDS = 4
SHARD_ID = 0     # Four workers: shard IDs 0, 1, 2, 3.
REUSE_EXISTING = True
assert MODE in ('worker', 'aggregate')
assert isinstance(N_SHARDS, int) and N_SHARDS > 0
assert isinstance(SHARD_ID, int) and 0 <= SHARD_ID < N_SHARDS


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for part in iter(lambda: handle.read(2**20), b''):
            h.update(part)
    return h.hexdigest()

def canonical_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, allow_nan=False,
                                    separators=(',', ':')).encode()).hexdigest()

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.partial')
    temp.write_text(json.dumps(obj, indent=2, allow_nan=False, default=str))
    os.replace(temp, path)

def write_csv(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.partial')
    pd.DataFrame(data).to_csv(temp, index=False)
    os.replace(temp, path)

def truth(series):
    return series.astype(str).str.strip().str.lower().isin(['true', '1', 'yes'])


In [ ]:
EXPECTED_SIGNATURE_ZIP_SHA256 = '8098755eeca1d564a4e5e004aa9bdfd0e16f187b703f21110fe517c45200eeab'
if sha256_file(SIGNATURE_INPUT_ZIP) != EXPECTED_SIGNATURE_ZIP_SHA256:
    raise ValueError('The original-signature input ZIP has changed; reconcile it before fitting.')
EMBEDDED_ROOT = WORK / 'original_inputs'
EMBEDDED_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SIGNATURE_INPUT_ZIP) as z:
    required = ['payload.json', 'original_scores.npz', 'Final_Refined_Signatures_Gemini.json']
    if sorted(z.namelist()) != sorted(required):
        raise ValueError('Unexpected original-input archive contents')
    for name in required:
        (EMBEDDED_ROOT / name).write_bytes(z.read(name))
PAYLOAD = json.loads((EMBEDDED_ROOT / 'payload.json').read_text())
display(pd.DataFrame(PAYLOAD['source_reconciliation']))


In [ ]:
def load_inputs(zip_path, scratch, payload):
    zip_path, scratch = Path(zip_path), Path(scratch)
    if not zip_path.is_file():
        raise FileNotFoundError('Upload PDAC_main_cohort_multiomics_inputs.zip to '
                                'MyDrive/new_dysfunction/multiomics_inputs/ and rerun this cell. '
                                f'Expected: {zip_path}')
    digest = sha256_file(zip_path)
    if digest != payload['input_zip_sha256']:
        raise ValueError('This is not the verified input ZIP. Keep the original ZIP unchanged; '
                         'do not substitute the 105-case export.')
    dest = scratch / ('inputs_' + digest[:12])
    needed = ['prepared/matrices/rna_tumor_main_cohort.npz',
              'prepared/matrices/protein_tumor_main_cohort.npz',
              'prepared/model_covariates.csv', 'prepared/main_cohort_127.csv',
              'prepared/tumor_plex_map.csv',
              'prepared/uploaded_export_membership_in_main_cohort.csv',
              'prepared/preparation_contract.json']
    with zipfile.ZipFile(zip_path) as archive:
        for suffix in needed:
            hits = [x for x in archive.namelist() if x.endswith('/' + suffix)]
            if len(hits) != 1:
                raise ValueError(f'Expected exactly one {suffix}: {hits}')
            target = dest / suffix
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists():
                target.write_bytes(archive.read(hits[0]))
    root = dest / 'prepared'
    cohort = pd.read_csv(root / 'model_covariates.csv')
    reference = pd.read_csv(root / 'main_cohort_127.csv')
    ids = cohort.base_sample_id.astype(str).to_numpy()
    assert len(ids) == 127 and len(set(ids)) == 127
    assert np.array_equal(ids, reference.base_sample_id.astype(str).to_numpy())
    matrices = {}
    for layer, filename in [('RNA', 'rna_tumor_main_cohort.npz'),
                            ('Protein', 'protein_tumor_main_cohort.npz')]:
        with np.load(root / 'matrices' / filename, allow_pickle=False) as a:
            values = a['values'].astype(float)
            feature_ids = a['feature_ids'].astype(str)
            assert np.array_equal(a['patient_ids'].astype(str), ids), layer
            assert values.shape == (len(feature_ids), 127), layer
            assert len(feature_ids) == len(set(feature_ids)), 'Duplicate gene IDs'
            assert not np.isinf(values).any(), 'Infinite measurements'
            matrices[layer] = {'values': values, 'feature_ids': feature_ids}
    plex = pd.read_csv(root / 'tumor_plex_map.csv').query("layer == 'protein'")
    assert plex.base_sample_id.is_unique and len(plex) == 127
    cohort = cohort.merge(plex[['base_sample_id', 'plex']], on='base_sample_id',
                          how='left', validate='one_to_one', sort=False)
    membership = pd.read_csv(root / 'uploaded_export_membership_in_main_cohort.csv')
    membership['in_export_105'] = truth(membership.in_uploaded_105_case_export)
    cohort = cohort.merge(membership[['base_sample_id', 'in_export_105']],
                          on='base_sample_id', how='left', validate='one_to_one', sort=False)
    purity = pd.DataFrame(payload['dna_purity_records'])
    cohort = cohort.merge(purity, on='base_sample_id', how='left',
                          validate='one_to_one', sort=False)
    assert np.array_equal(cohort.base_sample_id.astype(str).to_numpy(), ids)
    assert cohort[['bmi', 'age', 'sex_centered', 'plex', 'in_export_105']].notna().all().all()
    assert cohort.plex.nunique() == 24 and int(cohort.in_export_105.sum()) == 96
    for col in ['wes_purity', 'wgs_purity']:
        observed = cohort[col].dropna()
        assert observed.between(0, 1).all(), 'DNA purity must be a fraction in [0,1]'
    assert int(cohort.wes_purity.notna().sum()) == 96
    cohort['adenosquamous_recorded'] = truth(cohort.adenosquamous_recorded)
    cohort['stage_major'] = cohort.stage_major.fillna('').astype(str)
    print(f'Input verified: {len(cohort)} paired tumour patients; '
          f'{cohort.plex.nunique()} plexes; '
          f'{np.isfinite(matrices["Protein"]["values"]).all(1).sum():,} complete proteins.')
    return {'cohort': cohort, 'matrices': matrices, 'zip_sha256': digest,
            'prepared_root': root}


In [ ]:
def load_original_context(inputs, payload, embedded_root):
    root = Path(embedded_root)
    jp, sp = root/payload['signature_file'], root/'original_scores.npz'
    if sha256_file(jp) != payload['signature_sha256'] or sha256_file(sp) != payload['original_scores_sha256']:
        raise ValueError('Embedded original inputs failed their checksums.')
    inputs['signatures'] = json.loads(jp.read_text())
    with np.load(sp, allow_pickle=False) as z:
        original = {k:z[k].copy() for k in z.files}
    ids = inputs['cohort'].base_sample_id.astype(str).to_numpy()
    assert np.array_equal(ids, original['patient_ids'])
    inventory = pd.DataFrame(payload['original_features'])
    assert np.array_equal(inventory.global_feature.to_numpy(), original['feature_ids'])
    assert original['values'].shape == original['valid'].shape == (127,914)
    assert np.isfinite(original['values']).all() and np.isfinite(original['fractions']).all()
    np.testing.assert_allclose(original['values'].std(0,ddof=1),1,atol=1e-10)
    assert original['fractions'].shape == (127,26)
    assert ((original['fractions']>=0)&(original['fractions']<=1)).all()
    for j in range(26):
        inputs['cohort'][f'fraction_{j:02d}'] = original['fractions'][:,j]
        inputs['cohort'][f'valid_component_{j:02d}'] = original['component_totals'][:,j]>=1000
    assert np.array_equal(original['valid'].sum(0), inventory.original_n_patients.to_numpy())
    inputs.update(original=original, inventory=inventory, embedded_root=root)
    return inputs


In [ ]:
def score_programmes(inputs, payload):
    """Reproduce the gene-z/cap/mean rule. Never inspect BMI or archived effect values."""
    matrices, eligible, lookup = inputs['matrices'], {}, {}
    for layer in LAYERS:
        v, names = matrices[layer]['values'], matrices[layer]['feature_ids']
        complete = np.isfinite(v).all(1)
        good = np.zeros(len(names),bool)
        good[complete] = v[complete].std(1,ddof=1)>1e-8
        eligible[layer] = set(names[complete & good])
        lookup[layer] = {g:i for i,g in enumerate(names)}
    records, unique = [], {}
    for f in payload['original_features']:
        raw = inputs['signatures'][f['CellType']][f['Signature']]
        if not isinstance(raw,list) or not all(isinstance(x,str) for x in raw):
            raise ValueError('The frozen original schema is a list of unsigned gene symbols.')
        genes = sorted(set(x.strip().upper() for x in raw if x.strip()))
        original_hash = hashlib.sha256(('POS:'+','.join(genes)+'|NEG:').encode()).hexdigest()
        assert original_hash == f['source_gene_hash']
        shared = sorted(set(genes)&eligible['RNA']&eligible['Protein'])
        ok = len(shared)>=payload['scoring_min_genes'] and len(shared)/len(genes)>=payload['scoring_min_fraction']
        key = 'SIG_'+hashlib.sha256((';'.join(shared)).encode()).hexdigest()[:12] if ok else ''
        record = dict(f, programme=key, included=ok, source_genes=len(genes),
                      matched_complete_genes=len(shared), coverage_fraction=len(shared)/len(genes),
                      source_gene_list=';'.join(genes), matched_gene_list=';'.join(shared),
                      unavailable_gene_list=';'.join(sorted(set(genes)-set(shared))),
                      exclusion_reason='' if ok else 'fewer than 5 complete matched genes or below 50% coverage')
        records.append(record)
        if ok:
            if key in unique:
                assert unique[key]['matched_genes']==shared, 'Hash collision'
            else:
                unique[key]={'name':key,'matched_genes':shared,'links':[],
                             'reporting_role':'all_eligible_original_signatures'}
            unique[key]['links'].append({k:record[k] for k in ['family','CellType','Signature','global_feature',
                                          'source_gene_hash','source_genes','coverage_fraction']})
    # Stable order independent of outcomes and BMI. Freeze both original and measured duplicates.
    programmes = [unique[k] for k in sorted(unique)]
    Y, coherence, scalers, membership = [], [], [], []
    rng = np.random.default_rng(142901)
    for programme in programmes:
        name, genes = programme['name'], programme['matched_genes']
        rep = min(programme['links'],key=lambda r:(r['Signature'],r['family'],r['CellType']))
        programme['label'] = rep['Signature'].removesuffix('_Signature').replace('_',' ')+' · '+name[-4:]
        programme['n_source_definitions'] = len(set(x['source_gene_hash'] for x in programme['links']))
        vals = []
        for layer in LAYERS:
            a = matrices[layer]['values'][[lookup[layer][g] for g in genes]]
            mean, sd = a.mean(1), a.std(1,ddof=1)
            z = np.clip((a-mean[:,None])/sd[:,None],-payload['gene_z_cap'],payload['gene_z_cap'])
            score = z.mean(0)
            mu, sigma = float(score.mean()),float(score.std(ddof=1))
            if sigma<1e-8: raise ValueError(f'Constant score: {name} {layer}')
            vals.append((score-mu)/sigma)
            for gene,m,s in zip(genes,mean,sd):
                scalers.append(dict(programme=name,layer=layer,gene=gene,gene_mean_127=m,gene_sd_127=s,
                                    score_mean_127=mu,score_sd_127=sigma,gene_z_cap=payload['gene_z_cap']))
            c=np.corrcoef(z); mean_r=float(c[np.triu_indices(len(genes),1)].mean())
            halves=[]
            for _ in range(100):
                ix=rng.permutation(len(genes));cut=len(ix)//2
                halves.append(np.corrcoef(z[ix[:cut]].mean(0),z[ix[cut:]].mean(0))[0,1])
            coherence.append(dict(programme=name,layer=layer,mean_gene_gene_correlation=mean_r,
                                  median_split_gene_half_correlation=float(np.median(halves)),
                                  interpretation='Internal gene-set coherence, not technical reliability'))
        Y.append(np.column_stack(vals))
        membership.extend(dict(programme=name,gene=g,sign=1) for g in genes)
    mapping=pd.DataFrame(records)
    overlap=[]
    for i,a in enumerate(programmes):
        ga=set(a['matched_genes'])
        for b in programmes[i+1:]:
            gb=set(b['matched_genes']);j=len(ga&gb)/len(ga|gb)
            overlap.append(dict(programme_a=a['name'],programme_b=b['name'],matched_jaccard=j,
                                shared_genes=len(ga&gb),near_duplicate=j>=0.8))
    # RNA using the full available source signature is an explicitly separate linear coverage check.
    # One column per unique SOURCE list; it is never silently substituted for matched RNA.
    full_sets={};full_rna=[];full_names=[]
    for row in mapping[mapping.included].sort_values('source_gene_hash').to_dict('records'):
        key=row['source_gene_hash']
        if key in full_sets: continue
        genes=sorted(set(row['source_gene_list'].split(';'))&eligible['RNA'])
        a=matrices['RNA']['values'][[lookup['RNA'][g] for g in genes]]
        z=np.clip((a-a.mean(1)[:,None])/a.std(1,ddof=1)[:,None],-5,5).mean(0)
        full_sets[key]=len(full_names);full_names.append(key)
        full_rna.append((z-z.mean())/z.std(ddof=1))
    mapping['full_rna_index']=[full_sets.get(x,-1) for x in mapping.source_gene_hash]
    return dict(Y=np.stack(Y,axis=1),names=[p['name'] for p in programmes],programmes=programmes,
                coverage=mapping,coherence=pd.DataFrame(coherence),scalers=pd.DataFrame(scalers),
                membership=pd.DataFrame(membership),overlap=pd.DataFrame(overlap),
                full_rna=np.column_stack(full_rna),full_rna_names=full_names)


In [ ]:
inputs = load_inputs(INPUT_ZIP, WORK / 'scratch', PAYLOAD)
inputs = load_original_context(inputs, PAYLOAD, EMBEDDED_ROOT)
scores = score_programmes(inputs, PAYLOAD)
print('Original feature labels:', len(scores['coverage']))
print('Covered labels:', int(scores['coverage'].included.sum()))
print('Distinct measured scores:', len(scores['names']))
display(scores['coverage'].groupby('family').agg(
    original_features=('global_feature','size'), covered_features=('included','sum')))
display(scores['coverage'][['family','CellType','Signature','matched_complete_genes',
                             'source_genes','coverage_fraction','included','exclusion_reason']].head(20))


In [ ]:
VARIANTS = {'primary':('all','mundlak',None),'fixed_plex':('all','fixed',None),
 'export_subset_96':('export','mundlak',None),'wes_purity_96':('export','mundlak','wes_purity'),
 'stage_subset_base':('stage','mundlak',None),'stage_adjusted':('stage','mundlak','stage'),
 'BMI_at_least_18_5':('BMI_at_least_18_5','mundlak',None)}


In [ ]:
def subset_mask(cohort, subset):
    if subset=='all': return np.ones(len(cohort),bool)
    if subset=='export': return cohort.in_export_105.to_numpy(bool)
    if subset=='stage': return cohort.stage_major.isin(['I','II','III','IV']).to_numpy()
    if subset=='BMI_at_least_18_5': return cohort.bmi.ge(18.5).to_numpy()
    if subset.startswith('component_'): return cohort['valid_'+subset].to_numpy(bool)
    raise ValueError(subset)


In [ ]:
def plan_jobs(inputs, scores, payload, config):
    mapping=scores['coverage']; names=scores['names']
    selected=[]
    for t in payload['sensitivity_targets']:
        f=mapping[(mapping.family==t['family'])&(mapping.CellType==t['CellType'])&(mapping.Signature==t['Signature'])]
        if len(f)!=1 or not f.iloc[0].included: raise ValueError(f'Prespecified target failed coverage: {t}')
        selected.append(f.iloc[0].to_dict())
    scores['sensitivity_targets']=pd.DataFrame(selected)
    primary=names[:1] if config['PROFILE']=='smoke' else names
    panel=primary if config['PROFILE']=='smoke' else sorted(set(f['programme'] for f in selected))
    jobs=[dict(phase='primary',variant='primary',programme=p) for p in primary]
    for v in config['RUN_VARIANTS']:
        if v=='primary':continue
        if v=='BMI_at_least_18_5' and inputs['cohort'].bmi.ge(18.5).all():continue
        for p in panel:jobs.append(dict(phase='sensitivities',variant=v,programme=p))
    if config['RUN_COMPOSITION_SENSITIVITY'] and config['PROFILE']!='smoke':
        for k,f in enumerate(selected):
            for adjusted in [False,True]:
                v=('fraction_adjusted' if adjusted else 'fraction_subset_base')+f'__target_{k:02d}'
                comp=f'component_{int(f["component_index"]):02d}'
                extra=f'fraction_{int(f["component_index"]):02d}' if adjusted else None
                VARIANTS[v]=(comp,'mundlak',extra)
                jobs.append(dict(phase='composition',variant=v,programme=f['programme'],
                                 linked_global_feature=f['global_feature'],fraction_component=f['global_celltype']))
    return pd.DataFrame(jobs)


In [ ]:
def make_design(cohort, mask, adjustment='mundlak', extra=None):
    d=cohort.loc[mask].copy().reset_index(drop=True)
    levels=sorted(d.plex.unique());group=pd.Categorical(d.plex,categories=levels).codes
    raw={'bmi_5':d.bmi.to_numpy()/5,'age_10y':d.age.to_numpy()/10,'sex':d.sex_centered.to_numpy()}
    if extra=='wes_purity':
        if not d.wes_purity.notna().all():raise ValueError('Missing WES purity')
        raw['wes_purity_10pct']=d.wes_purity.to_numpy()/0.1
    elif extra=='stage':
        for stage in ['II','III','IV']:raw['stage_'+stage]=d.stage_major.eq(stage).to_numpy(float)
    elif extra and extra.startswith('fraction_'):
        # A single RNA-derived contribution covariate, not protein deconvolution or a full composition.
        clipped=np.clip(cohort[extra].to_numpy(),1e-6,1-1e-6)
        logit=np.log(clipped/(1-clipped));sd=logit.std(ddof=1)
        if sd<1e-8: raise ValueError('Constant fraction covariate')
        raw[extra+'_logit_z']=(logit[mask]-logit.mean())/sd
    names,columns,between,dropped=[],[],[],[]
    for key,x in raw.items():
        mean=pd.Series(x).groupby(group).transform('mean').to_numpy()
        terms=[(key+'_within',x-mean)]
        if adjustment=='mundlak':between.append((key+'_between',mean-x.mean()))
        for name,value in terms:
            if np.std(value)<1e-10:
                if name=='bmi_5_within':raise ValueError('No within-plex BMI variation')
                dropped.append(name+' (constant)');continue
            names.append(name);columns.append(value)
    for name,value in between:
        if np.std(value)<1e-10:dropped.append(name+' (constant)');continue
        names.append(name);columns.append(value)
    X=np.column_stack(columns);assert names[0]=='bmi_5_within'
    if not np.isfinite(X).all():raise ValueError('Non-finite design')
    Q=helmert(len(levels),full=False).T if adjustment=='fixed' else None
    full=np.column_stack([np.ones(len(d)),X] + ([Q[group]] if Q is not None else []))
    if np.linalg.matrix_rank(full)!=full.shape[1]:raise ValueError('Rank-deficient design; no automatic covariate deletion')
    return dict(data=d,X=X,names=names,group=group,levels=levels,Q=Q,adjustment=adjustment,extra=extra,
                dropped_constant_terms=dropped)


In [ ]:
def build_model(design, observed, config):
    import pymc as pm
    import pytensor.tensor as pt
    coords = {'patient': design['data'].base_sample_id.tolist(), 'layer': LAYERS,
              'term': design['names'], 'plex': design['levels']}
    if design['adjustment'] == 'fixed':
        coords['plex_contrast'] = list(range(len(design['levels'])-1))
    prior = np.full(len(design['names']), config['COVARIATE_PRIOR_SD'])
    for i, name in enumerate(design['names']):
        if name.startswith('bmi_5_'): prior[i] = config['BMI_PRIOR_SD']
    with pm.Model(coords=coords) as model:
        alpha = pm.Normal('alpha', 0, 0.5, dims='layer')
        coefficient = pm.Normal('coefficient', 0, prior[:, None], dims=('term', 'layer'))
        if design['adjustment'] == 'mundlak':
            group_scale = pm.HalfNormal('group_scale', config['GROUP_SD_PRIOR'], dims='layer')
            group_z = pm.Normal('group_z', 0, 1, dims=('plex', 'layer'))
            group_effect = pm.Deterministic('group_effect', group_z*group_scale,
                                           dims=('plex', 'layer'))
        else:
            group_coefficient = pm.Normal('group_coefficient', 0,
                                           config['FIXED_GROUP_PRIOR_SD'],
                                           dims=('plex_contrast', 'layer'))
            group_effect = pm.Deterministic('group_effect',
                pt.dot(design['Q'], group_coefficient), dims=('plex', 'layer'))
        mu = alpha + pt.dot(design['X'], coefficient) + group_effect[design['group']]
        chol, corr, scales = pm.LKJCholeskyCov('resid_chol', n=2, eta=2,
            sd_dist=pm.HalfNormal.dist(1.0, shape=2), compute_corr=True)
        nu_minus_two = pm.Exponential('nu_minus_two', 0.1)
        nu = pm.Deterministic('nu', 2+nu_minus_two)
        pm.Deterministic('bmi_effect_per5', coefficient[0], dims='layer')
        pm.Deterministic('protein_minus_RNA_per5', coefficient[0,1]-coefficient[0,0])
        pm.Deterministic('residual_correlation', corr[0,1])
        pm.MvStudentT('paired_score', nu=nu, mu=mu, chol=chol,
                      observed=observed, dims=('patient', 'layer'))
    return model


In [ ]:
def validate_prior(inputs, scores, config, run):
    import pymc as pm
    design = make_design(inputs['cohort'], np.ones(127, bool))
    model = build_model(design, scores['Y'][:,0,:], config)
    initial = float(model.compile_logp()(model.initial_point()))
    if not np.isfinite(initial): raise RuntimeError('Non-finite initial log density')
    with model:
        pred = pm.sample_prior_predictive(samples=500, random_seed=config['SEED'])
    a = pred.prior_predictive['paired_score'].values
    if a.shape[-2:] != (127, 2) or not np.isfinite(a).all():
        raise RuntimeError('Prior predictive shape or finite-value check failed')
    stats = {'initial_log_density_finite': True, 'prior_predictive_draws': 500,
             'absolute_score_quantiles': dict(zip(['50%', '90%', '95%', '99%'],
                [float(v) for v in np.quantile(np.abs(a), [.5,.9,.95,.99])])),
             'scope': 'Computation/scale check; not simulation validation of interval calibration'}
    write_json(run/'prior_predictive_checks.json', stats)
    return stats


In [ ]:
def posterior_array(idata, name):
    da = idata.posterior[name].stack(sample=('chain', 'draw'))
    return da.transpose('sample', *[d for d in da.dims if d != 'sample']).values


In [ ]:
def hdi(values, probability=0.95):
    a = np.sort(np.asarray(values, float).ravel())
    if not np.isfinite(a).all() or len(a) < 20:
        raise ValueError('Cannot summarize incomplete posterior draws')
    width = int(np.floor(probability*len(a)))
    left = int(np.argmin(a[width:]-a[:-width]))
    return float(a[left]), float(a[left+width])


In [ ]:
def effect_summary(idata, programme, variant, design, role, numerical_pass):
    beta = posterior_array(idata, 'bmi_effect_per5')
    quantities = [('RNA_BMI_per5', beta[:,0]), ('Protein_BMI_per5', beta[:,1]),
                  ('Protein_minus_RNA_BMI_per5', beta[:,1]-beta[:,0]),
                  ('residual_RNA_Protein_correlation',
                   posterior_array(idata, 'residual_correlation'))]
    result = []
    for quantity, values in quantities:
        lo, hi = hdi(values)
        result.append({'programme': programme, 'variant': variant, 'quantity': quantity,
            'posterior_mean': float(np.mean(values)), 'posterior_median': float(np.median(values)),
            'hdi_95_lower': lo, 'hdi_95_upper': hi, 'n_patients': len(design['data']),
            'n_plexes': len(design['levels']), 'numerical_checks_pass': bool(numerical_pass),
            'reporting_role': role, 'effect_units': ('correlation' if quantity.startswith('residual')
                else '127-patient programme-score SD per 5 kg/m2, within plex'),
            'interval_type': '95% highest-density posterior interval'})
    return pd.DataFrame(result)


In [ ]:
def numerical_diagnostics(idata, model, config, folder):
    import arviz as az
    names = [v.name for v in model.free_RVs]
    summary = az.summary(idata, var_names=names, kind='diagnostics', round_to='none')
    summary.rename_axis('parameter').reset_index().to_csv(folder/'parameter_diagnostics.csv', index=False)
    columns = summary[['r_hat', 'ess_bulk', 'ess_tail']].to_numpy(float)
    finite = bool(np.isfinite(columns).all())
    rhat = float(np.nanmax(summary.r_hat))
    bulk, tail = float(np.nanmin(summary.ess_bulk)), float(np.nanmin(summary.ess_tail))
    stats = idata.sample_stats
    divergences = int(stats['diverging'].sum())
    bfmi = float(np.nanmin(az.bfmi(idata)))
    if 'reached_max_treedepth' in stats:
        depth_hits = int(stats['reached_max_treedepth'].sum())
    elif 'tree_depth' in stats:
        depth_hits = int((stats.tree_depth >= config['MAX_TREEDEPTH']).sum())
    elif 'depth' in stats:
        depth_hits = int((stats.depth >= config['MAX_TREEDEPTH']).sum())
    else:
        raise RuntimeError('Sampler returned no tree-depth statistic; cannot verify the gate')
    passed = finite and rhat <= 1.01 and min(bulk, tail) >= 400 and bfmi >= 0.30 \
             and divergences == 0 and depth_hits == 0
    return {'max_rhat': rhat if np.isfinite(rhat) else None,
            'min_ess_bulk': bulk if np.isfinite(bulk) else None,
            'min_ess_tail': tail if np.isfinite(tail) else None,
            'min_bfmi': bfmi if np.isfinite(bfmi) else None,
            'divergences': divergences, 'maximum_depth_hits': depth_hits,
            'all_diagnostics_finite': finite, 'numerical_checks_pass': bool(passed),
            'publication_profile': config['PROFILE'] == 'publication'}


In [ ]:
def compute_ppc(idata, design, observed, config, folder, seed):
    rng = np.random.default_rng(seed+231)
    a, c, g = [posterior_array(idata, n) for n in ['alpha', 'coefficient', 'group_effect']]
    total = len(a)
    chosen = np.sort(rng.choice(total, min(config['PPC_DRAWS'], total), replace=False))
    mu = a[chosen,None,:] + np.einsum('nt,stl->snl', design['X'], c[chosen]) \
         + g[chosen][:, design['group'], :]
    scales = posterior_array(idata, 'resid_chol_stds')[chosen]
    corr = posterior_array(idata, 'resid_chol_corr')[chosen]
    nu = posterior_array(idata, 'nu')[chosen]
    S = corr*scales[:,:,None]*scales[:,None,:]
    L = np.linalg.cholesky(S)
    z = rng.normal(size=mu.shape)
    gaussian = np.einsum('sij,snj->sni', L, z)
    w = np.sqrt(rng.chisquare(nu[:,None], size=mu.shape[:2])/nu[:,None])
    replicate = mu + gaussian/w[:,:,None]
    observed_residual = observed[None,:,:]-mu
    replicate_residual = replicate-mu
    checks = []
    def add(name, obs, rep):
        upper = float(np.mean(rep >= obs))
        checks.append({'check': name, 'observed_draw_median': float(np.median(obs)),
                       'replicated_median': float(np.median(rep)),
                       'replicated_95_lower': float(np.quantile(rep,.025)),
                       'replicated_95_upper': float(np.quantile(rep,.975)),
                       'posterior_predictive_upper_tail_probability': upper,
                       'flag': bool(upper < .025 or upper > .975),
                       'interpretation': 'Descriptive predictive check; not a calibrated p-value'})
    for j, layer in enumerate(LAYERS):
        add(layer+'_residual_SD', observed_residual[:,:,j].std(1,ddof=1),
            replicate_residual[:,:,j].std(1,ddof=1))
        add(layer+'_maximum_absolute_residual', np.abs(observed_residual[:,:,j]).max(1),
            np.abs(replicate_residual[:,:,j]).max(1))
    def corrs(x):
        v=x-x.mean(1,keepdims=True)
        return np.sum(v[:,:,0]*v[:,:,1],axis=1)/np.sqrt(
            np.sum(v[:,:,0]**2,axis=1)*np.sum(v[:,:,1]**2,axis=1))
    add('RNA_Protein_residual_correlation', corrs(observed_residual), corrs(replicate_residual))
    low, high = np.quantile(replicate,[.05,.95],axis=0)
    for j, layer in enumerate(LAYERS):
        checks.append({'check': layer+'_90pct_in_sample_predictive_coverage',
                       'observed_draw_median': float(np.mean((observed[:,j]>=low[:,j]) &
                                                            (observed[:,j]<=high[:,j]))),
                       'interpretation': 'In-sample descriptive coverage; not out-of-sample validation',
                       'flag': False})
    write_csv(folder/'predictive_checks.csv', checks)
    write_csv(folder/'patient_residuals.csv', pd.DataFrame({
        'base_sample_id': design['data'].base_sample_id,
        'RNA_residual': observed[:,0]-mu[:,:,0].mean(0),
        'Protein_residual': observed[:,1]-mu[:,:,1].mean(0)}))
    return int(sum(bool(x.get('flag',False)) for x in checks))


In [ ]:
def compute_loo(idata, design, folder):
    import arviz as az
    ll = idata.log_likelihood['paired_score']
    point_dims = [d for d in ll.dims if d not in ['chain', 'draw']]
    if len(point_dims)!=1 or ll.sizes[point_dims[0]]!=len(design['data']):
        raise RuntimeError('LOO must have one joint RNA/protein likelihood per patient.')
    loo = az.loo(idata, pointwise=True, var_name='paired_score')
    k = np.asarray(loo.pareto_k).ravel()
    threshold = float(getattr(loo, 'good_k', .7))
    rows = pd.DataFrame({'base_sample_id': design['data'].base_sample_id,
                         'pareto_k': k, 'threshold': threshold,
                         'flag_for_refit': (~np.isfinite(k)) | (k>threshold)})
    write_csv(folder/'PSIS_LOO_patients.csv', rows)
    write_json(folder/'PSIS_LOO_summary.json', {
        'elpd_loo': float(loo.elpd_loo), 'se': float(loo.se), 'p_loo': float(loo.p_loo),
        'warning': bool(loo.warning), 'flagged_patients': int(rows.flag_for_refit.sum()),
        'unit': 'Patient: both RNA and protein held out together',
        'prediction_target': 'Another patient in an observed plex; not a new plex'})
    return rows


In [ ]:
def sample_model(model, config, seed, tune, draws):
    import pymc as pm
    if config['SAMPLER'] != 'pymc':
        raise ValueError("This notebook uses the tested PyMC NUTS sampler; set SAMPLER='pymc'.")
    with model:
        idata = pm.sample(draws=draws, tune=tune, chains=config['CHAINS'],
            cores=config['CORES'], random_seed=seed, target_accept=config['TARGET_ACCEPT'],
            nuts_sampler='pymc', nuts={'max_treedepth':config['MAX_TREEDEPTH']},
            mp_ctx='forkserver',                      # start clean workers instead of forking the kernel
            return_inferencedata=True, idata_kwargs={'log_likelihood':True},
            progressbar=True, compute_convergence_checks=True)
        if 'log_likelihood' not in idata.groups():
            idata = pm.compute_log_likelihood(idata, model=model, var_names=['paired_score'],
                extend_inferencedata=True, progressbar=False)
    return idata, 'pymc', None


In [ ]:
def run_one(inputs, scores, config, run, programme_index, variant, leave_out=None):
    import arviz as az
    name = scores['names'][programme_index]
    subset, adjustment, extra = VARIANTS[variant]
    mask = subset_mask(inputs['cohort'], subset)
    if leave_out is not None:
        mask &= inputs['cohort'].base_sample_id.ne(leave_out).to_numpy()
    design = make_design(inputs['cohort'], mask, adjustment, extra)
    observed = scores['Y'][mask,programme_index,:]
    job_variant = variant if leave_out is None else variant+'__without_'+str(leave_out)
    folder = run/'fits'/job_variant/name
    folder.mkdir(parents=True, exist_ok=True)
    status_file = folder/'status.json'
    if status_file.exists():
        previous = json.loads(status_file.read_text())
        if previous.get('postprocessing_complete') and not (
                config.get('FORCE_REFIT_FAILED',False) and not previous.get('numerical_checks_pass')):
            print('Resume: already finished', job_variant, name)
            return previous
    token = f'{name}:{job_variant}:{config["SEED"]}'
    seed = int(hashlib.sha256(token.encode()).hexdigest()[:8],16) % (2**31-1)
    model = build_model(design, observed, config)
    table = pd.DataFrame(design['X'], columns=design['names'])
    table.insert(0,'base_sample_id',design['data'].base_sample_id)
    write_csv(folder/'design_matrix.csv', table)
    write_json(folder/'design_notes.json',{'dropped_constant_terms':design.get('dropped_constant_terms',[])})
    base = {'programme': name, 'variant': job_variant, 'n_patients': len(observed),
            'n_plexes': len(design['levels']), 'completed': False,
            'postprocessing_complete': False, 'numerical_checks_pass': False}
    t0 = time.monotonic()
    for attempt in range(2 if config['RETRY_ONCE'] else 1):
        draws, tune = config['DRAWS']*(2**attempt), config['TUNE']*(2**attempt)
        trace = folder/f'posterior_attempt_{attempt+1}.nc'
        manifest = folder/f'attempt_{attempt+1}.json'
        if trace.exists() and not config.get('FORCE_REFIT_FAILED',False):
            idata = az.from_netcdf(trace)
            idata.load()
            idata.close()
            info = json.loads(manifest.read_text()) if manifest.exists() else {}
            backend, reason = info.get('sampler','resumed'), info.get('fallback_reason')
        else:
            print(f'Fit: {job_variant} | {name} | n={len(observed)} | '
                  f'{config["CHAINS"]} chains, {tune} warmup + {draws} retained | attempt {attempt+1}')
            idata, backend, reason = sample_model(model, config, seed+attempt*1009, tune, draws)
            temp = trace.with_name(trace.name+'.partial.nc')
            idata.to_netcdf(temp)
            os.replace(temp, trace)
            write_json(manifest, {'seed': seed+attempt*1009, 'sampler': backend,
                                 'fallback_reason': reason, 'tune': tune, 'draws': draws})
        diagnostics = numerical_diagnostics(idata, model, config, folder)
        if diagnostics['numerical_checks_pass'] or config['PROFILE']=='smoke': break
        print('Numerical thresholds not met; one longer attempt will be used if available.')
    role = scores['programmes'][programme_index]['reporting_role']
    effect_summary(idata,name,job_variant,design,role,diagnostics['numerical_checks_pass']).to_csv(
        folder/'effects_95HDI.csv',index=False)
    status = {**base, **diagnostics, 'completed': True, 'sampling_attempt': attempt+1,
              'sampler': backend, 'postprocessing_complete': False,
              'posterior_file': trace.name, 'sampling_seconds_this_session': time.monotonic()-t0}
    write_json(status_file,status)
    status['predictive_check_flags'] = compute_ppc(idata,design,observed,config,folder,seed)
    if variant=='primary' and leave_out is None:
        loo = compute_loo(idata,design,folder)
        status['PSIS_flagged_patients'] = int(loo.flag_for_refit.sum())
    status['postprocessing_complete'] = True
    status['publication_numerical_gate_pass'] = bool(
        diagnostics['numerical_checks_pass'] and config['PROFILE']=='publication')
    write_json(status_file,status)
    del idata
    return status


In [ ]:
def aggregate(run):
    states=[json.loads(p.read_text()) for p in sorted((run/'fits').glob('*/*/status.json'))]
    if states: write_csv(run/'results/run_status.csv',states)
    effects=[]; checks=[]
    for p in sorted((run/'fits').glob('*/*/effects_95HDI.csv')):
        effects.append(pd.read_csv(p))
    for p in sorted((run/'fits').glob('*/*/predictive_checks.csv')):
        df=pd.read_csv(p);df.insert(0,'programme',p.parent.name);df.insert(1,'variant',p.parent.parent.name)
        checks.append(df)
    if effects: write_csv(run/'results/all_effects_95HDI.csv',pd.concat(effects,ignore_index=True))
    if checks: write_csv(run/'results/all_predictive_checks.csv',pd.concat(checks,ignore_index=True))
    return pd.DataFrame(states)


In [ ]:
def run_phase(inputs, scores, config, run, phase):
    states=[]
    for job in scores['jobs'].query('phase == @phase').to_dict('records'):
        k=scores['names'].index(job['programme']);v=job['variant']
        try:state=run_one(inputs,scores,config,run,k,v)
        except Exception as exc:
            folder=run/'fits'/v/job['programme'];folder.mkdir(parents=True,exist_ok=True)
            (folder/'error.txt').write_text(traceback.format_exc())
            state=dict(programme=job['programme'],variant=v,completed=False,postprocessing_complete=False,
                       numerical_checks_pass=False,error=f'{type(exc).__name__}: {exc}')
            write_json(folder/'status.json',state)
            if config.get('STOP_ON_FIT_ERROR',True):raise
            print('Fit error saved; continuing:',state['error'][:200])
        states.append(state);aggregate(run)
        if phase=='primary' and len(states)==1 and state.get('sampling_seconds_this_session',0)>0:
            estimate=state['sampling_seconds_this_session']*len(scores['jobs'])/3600
            print(f'Rough sequential runtime from first fit: {estimate:.1f} h, excluding retries/influence. '
                  'This varies by signature and runtime. Every finished fit is checkpointed.')
    return pd.DataFrame(states)


In [ ]:
def run_influence(inputs,scores,config,run):
    if not config['RUN_INFLUENCE_REFITS']:
        print('Deletion refits disabled; unresolved PSIS flags remain in reports.');return pd.DataFrame()
    states=[];names=scores['names'][:1] if config['PROFILE']=='smoke' else scores['names']
    for name in names:
        p=run/'fits'/'primary'/name/'PSIS_LOO_patients.csv'
        if not p.exists():continue
        flags=pd.read_csv(p);flags=flags[truth(flags.flag_for_refit)].sort_values('pareto_k',ascending=False)
        for pid in flags.base_sample_id:
            try:state=run_one(inputs,scores,config,run,scores['names'].index(name),'primary',leave_out=pid)
            except Exception as exc:
                folder=run/'fits'/('primary__without_'+str(pid))/name;folder.mkdir(parents=True,exist_ok=True)
                (folder/'error.txt').write_text(traceback.format_exc())
                state=dict(programme=name,variant='primary__without_'+str(pid),completed=False,
                           postprocessing_complete=False,numerical_checks_pass=False,error=str(exc))
                write_json(folder/'status.json',state)
                if config.get('STOP_ON_FIT_ERROR',True):raise
            states.append(state);aggregate(run)
    return pd.DataFrame(states)


In [ ]:
def within_plex_bmi_weights(raw, group, ix):
    """FWL coefficient weights: exact OLS BMI slope adjusted for age, sex and fixed plex."""
    x=raw[ix];g=group[ix];levels=int(group.max())+1
    count=np.bincount(g,minlength=levels);sums=np.zeros((levels,raw.shape[1]))
    np.add.at(sums,g,x)
    means=np.divide(sums,count[:,None],out=np.zeros_like(sums),where=count[:,None]>0)
    w=x-means[g]
    if np.linalg.matrix_rank(w)<raw.shape[1]:return None
    contrast=np.zeros(raw.shape[1]);contrast[0]=1
    return w@np.linalg.solve(w.T@w,contrast)


In [ ]:
def concordance_statistics(a,b,weights):
    a=np.asarray(a,float);b=np.asarray(b,float)
    if a.ndim==1:a=a[None,:];b=b[None,:]
    valid=np.isfinite(a)&np.isfinite(b)
    w=valid*np.asarray(weights)[None,:];den=w.sum(1)
    aa=np.where(valid,a,0);bb=np.where(valid,b,0)
    agreement=np.divide((w*(aa*bb>0)).sum(1),den,out=np.full(len(a),np.nan),where=den>0)
    ma=np.divide((w*aa).sum(1),den,out=np.zeros(len(a)),where=den>0)
    mb=np.divide((w*bb).sum(1),den,out=np.zeros(len(a)),where=den>0)
    da=aa-ma[:,None];db=bb-mb[:,None]
    cov=(w*da*db).sum(1);v=np.sqrt((w*da**2).sum(1)*(w*db**2).sum(1))
    corr=np.divide(cov,v,out=np.full(len(a),np.nan),where=v>1e-12)
    return dict(sign_agreement=agreement,correlation_of_estimated_slopes=corr)


In [ ]:
def bootstrap_two_stage(inputs,scores,config,run):
    """Paired patient bootstrap of a fixed-plex LINEAR benchmark, including cell-type scores.

    The original hierarchical models are not refitted. Frozen deconvolution scores/scaling
    and the observed plexes are conditioned upon. This is not posterior calibration.
    """
    folder=run/'bootstrap';folder.mkdir(parents=True,exist_ok=True)
    marker=folder/'bootstrap_complete.json'
    if marker.exists():
        print('Resume: joint bootstrap benchmark complete');return pd.read_csv(folder/'concordance_95CI.csv')
    P=1 if config['PROFILE']=='smoke' else len(scores['names'])
    names=scores['names'][:P];name_index={p:i for i,p in enumerate(names)}
    eligible=scores['coverage'].included & scores['coverage'].programme.isin(names)
    links=scores['coverage'].loc[eligible].copy().reset_index().rename(columns={'index':'original_feature_index'})
    L=len(links);fi=links.original_feature_index.to_numpy(int)
    target=links.programme.map(name_index).to_numpy(int)
    cohort=inputs['cohort'];group=pd.Categorical(cohort.plex, categories=sorted(cohort.plex.unique())).codes
    raw=np.column_stack([cohort.bmi.to_numpy()/5,cohort.age.to_numpy()/10,cohort.sex_centered.to_numpy()])
    blocks=[np.flatnonzero(group==g) for g in np.unique(group)]
    original=inputs['original'];cell=original['values'][:,fi]
    B=config['BOOTSTRAP_REPLICATES'];rng=np.random.default_rng(config['SEED']+701)
    ix_draws=np.asarray([np.concatenate([rng.choice(b,len(b),replace=True) for b in blocks]) for _ in range(B)])
    bulk=np.full((B+1,P,2),np.nan);cell_beta=np.full((B+1,L),np.nan)
    paired_bulk=np.full((B+1,L,2),np.nan);full=np.full((B+1,scores['full_rna'].shape[1]),np.nan)
    component_groups=[]
    for comp,g in links.groupby('component_index',sort=True):
        cols=g.index.to_numpy(int);valid=original['component_totals'][:,int(comp)]>=1000
        assert (original['valid'][:,fi[cols]]==valid[:,None]).all()
        component_groups.append((cols,valid))
    for r,ix in enumerate([np.arange(127),*ix_draws]):
        w=within_plex_bmi_weights(raw,group,ix)
        if w is not None:
            bulk[r]=(w@scores['Y'][ix,:P,:].reshape(len(ix),-1)).reshape(P,2)
            full[r]=w@scores['full_rna'][ix]
        for cols,valid in component_groups:
            sub=ix[valid[ix]];ww=within_plex_bmi_weights(raw,group,sub)
            if ww is None:continue
            cell_beta[r,cols]=ww@cell[sub][:,cols]
            bb=(ww@scores['Y'][sub,:P,:].reshape(len(sub),-1)).reshape(P,2)
            paired_bulk[r,cols]=bb[target[cols]]
        if r and r%250==0:print('Joint bootstrap:',r,'/',B)
    nvalid=np.isfinite(cell_beta[1:]).sum(0)
    if (nvalid<.95*B).any() or np.isfinite(bulk[1:]).all((1,2)).sum()<.95*B:
        raise RuntimeError('Too many non-estimable resamples; inspect plex imbalance. No intervals finalized.')
    np.savez_compressed(folder/'joint_patient_bootstrap.npz',patient_draw_indices=ix_draws.astype(np.uint16),
                        patient_ids=cohort.base_sample_id.to_numpy(str),bulk_slopes=bulk,celltype_slopes=cell_beta,
                        paired_bulk_slopes=paired_bulk,full_rna_slopes=full,
                        programme_ids=np.asarray(names,dtype=str),feature_ids=links.global_feature.to_numpy(str))
    write_csv(folder/'celltype_bootstrap_feature_map.csv',links)
    rows=[]
    def interval_row(values,metadata):
        a=np.asarray(values);finite=np.isfinite(a[1:]);v=a[1:][finite]
        if not np.isfinite(a[0]) or len(v)<.95*B:
            lo=hi=float('nan')
        else:lo,hi=np.quantile(v,[.025,.975])
        return dict(metadata,estimate=float(a[0]),ci_95_lower=float(lo),ci_95_upper=float(hi),
                    valid_bootstrap_replicates=len(v),interval_type='95% percentile bootstrap confidence interval')
    for k,name in enumerate(names):
        for q,a in [('RNA_BMI_per5',bulk[:,k,0]),('Protein_BMI_per5',bulk[:,k,1]),
                    ('Protein_minus_RNA_BMI_per5',bulk[:,k,1]-bulk[:,k,0])]:
            rows.append(interval_row(a,dict(programme=name,quantity=q,n_patients=127)))
    write_csv(folder/'bulk_linear_effects_95CI.csv',rows)
    cellrows=[]
    for k,row in links.iterrows():
        metadata={key:row[key] for key in ['global_feature','family','CellType','Signature','programme','original_n_patients']}
        for q,a in [('celltype_BMI_per5',cell_beta[:,k]),('paired_RNA_BMI_per5',paired_bulk[:,k,0]),
                    ('paired_Protein_BMI_per5',paired_bulk[:,k,1])]:
            cellrows.append(interval_row(a,dict(metadata,quantity=q)))
    write_csv(folder/'celltype_and_matched_bulk_linear_95CI.csv',cellrows)
    # Approximate Gaussian summary-likelihood benchmark remains bivariate per programme.
    stage2=[]
    for k,name in enumerate(names):
        a=bulk[1:,k,:];a=a[np.isfinite(a).all(1)]
        S=np.cov(a,rowvar=False,ddof=1);v,U=np.linalg.eigh(S)
        floor=max(float(v.max())*1e-10,1e-12)
        S=U@np.diag(np.maximum(v,floor))@U.T;prec=np.linalg.inv(S)
        C=np.linalg.inv(prec+np.eye(2)/config['BMI_PRIOR_SD']**2);m=C@prec@bulk[0,k]
        for q,c in [('RNA_BMI_per5',[1,0]),('Protein_BMI_per5',[0,1]),('Protein_minus_RNA_BMI_per5',[-1,1])]:
            c=np.asarray(c);mu=float(c@m);sd=float(np.sqrt(c@C@c))
            stage2.append(dict(programme=name,quantity=q,posterior_mean=mu,
                hdi_95_lower=mu-norm.ppf(.975)*sd,hdi_95_upper=mu+norm.ppf(.975)*sd,
                covariance_eigenvalue_floor=floor,
                interval_type='95% HDI of Gaussian summary-likelihood approximation',
                scope='Fixed-plex linear benchmark, not the Student-t posterior or its calibration'))
    write_csv(folder/'two_stage_gaussian_95HDI.csv',stage2)
    # Export the full dependence, not just each programme's two-dimensional covariance.
    joint=np.column_stack([bulk[1:].reshape(B,-1),cell_beta[1:]])
    complete=np.isfinite(joint).all(1)
    np.savez_compressed(folder/'joint_slope_covariance.npz',covariance=np.cov(joint[complete],rowvar=False),
        labels=np.asarray([p+'__'+l for p in names for l in LAYERS]+links.global_feature.tolist(),dtype=str),
        complete_replicates=int(complete.sum()))
    # A coverage-only bridge: full-source RNA and measured-source RNA use the same patients/draws.
    bridge=[]
    for _,f in links.drop_duplicates('source_gene_hash').iterrows():
        k=name_index[f.programme];j=int(f.full_rna_index)
        bridge.append(interval_row(bulk[:,k,0]-full[:,j],dict(source_gene_hash=f.source_gene_hash,
            programme=f.programme,quantity='matched_minus_full_RNA_BMI_per5',
            interpretation='Different gene sets and outcome SDs; descriptive coverage sensitivity')))
    write_csv(folder/'RNA_gene_coverage_sensitivity_95CI.csv',bridge)
    concordance=[]
    def summarize(a,b,w,meta,unique_count):
        if unique_count<1:return
        stats=concordance_statistics(a,b,w)
        for metric,x in stats.items():
            if metric.startswith('correlation') and unique_count<3:continue
            concordance.append(interval_row(x,dict(meta,metric=metric,unique_programmes=unique_count,
                compared_feature_labels=len(w),weighting='One total weight per unique measured programme within this group')))
    summarize(bulk[:,:,0],bulk[:,:,1],np.ones(P),dict(scope='bulk',family='',CellType='',stratum='all',
                comparison='RNA_vs_Protein'),P)
    groups=[('family',fam,'',g) for fam,g in links.groupby('family',sort=True)]
    groups += [('celltype',fam,cellname,g) for (fam,cellname),g in links.groupby(['family','CellType'],sort=True)]
    for scope,fam,cellname,g in groups:
        for stratum in ['all','archived_HDI_excludes_zero','archived_HDI_includes_zero']:
            gg=g if stratum=='all' else g[g.original_hdi_excludes_zero.eq(stratum.endswith('excludes_zero'))]
            if gg.empty:continue
            cols=gg.index.to_numpy();w=(1/gg.groupby('programme').programme.transform('size')).to_numpy()
            for layer,j in [('RNA',0),('Protein',1)]:
                summarize(cell_beta[:,cols],paired_bulk[:,cols,j],w,
                    dict(scope=scope,family=fam,CellType=cellname,stratum=stratum,
                         comparison='celltype_vs_'+layer),gg.programme.nunique())
    write_csv(folder/'concordance_95CI.csv',concordance)
    write_json(marker,dict(complete=True,bootstrap_replicates=B,
        resampling='Same paired patient indices within plex for all cell-type, RNA and protein scores',
        original_low_read_masks_preserved=True,original_Bayesian_model_refitted=False,
        benchmark='OLS adjusted for age, sex and fixed plex; cell/bulk comparisons use the same valid patients',
        original_effect_tables='Used only as descriptive reference and fixed, post hoc HDI strata',
        weighting='Unique measured lists receive equal total weight within a reported cell type/family',
        concordance_inference='Descriptive agreement of estimated slopes, not a test above chance or independent replication',
        no_signature_resampling=True,between_programme_dependence_retained=True,
        conditions_on_frozen_scores_and_deconvolution=True,conditions_on_observed_plexes=True,
        interval_calibration_of_StudentT_or_original_BHM_established=False))
    return pd.DataFrame(concordance)


In [ ]:
def linked_results(scores, run):
    status=aggregate(run);p=run/'results/all_effects_95HDI.csv'
    if not p.exists():return pd.DataFrame()
    effects=pd.read_csv(p);primary=effects[effects.variant.eq('primary')].copy()
    linked=scores['coverage'].copy()
    for quantity,g in primary.groupby('quantity',sort=True):
        cols=['programme','posterior_mean','hdi_95_lower','hdi_95_upper','numerical_checks_pass']
        g=g[cols].rename(columns={c:quantity+'__'+c for c in cols[1:]})
        linked=linked.merge(g,on='programme',how='left',validate='many_to_one')
    if len(status):
        meta=status[status.variant.eq('primary')]
        cols=[c for c in ['programme','predictive_check_flags','PSIS_flagged_patients','postprocessing_complete'] if c in meta]
        linked=linked.merge(meta[cols],on='programme',how='left',validate='many_to_one')
    linked['interpretation']='Archived cell-type and new bulk estimates have distinct outcomes/designs; no joint Bayesian contrast is inferred'
    write_csv(run/'results/all_original_features_linked_95HDI.csv',linked)
    write_csv(run/'results/unique_bulk_programme_effects_95HDI.csv',primary)
    write_csv(run/'results/sensitivity_effects_95HDI.csv',effects[~effects.variant.eq('primary')])
    if len(primary):
        plot=primary.pivot(index='programme',columns='quantity',values='posterior_mean').reset_index()
        qc=primary.groupby('programme').numerical_checks_pass.apply(lambda x:truth(x).all())
        plot['numerical_checks_pass']=plot.programme.map(qc)
        write_csv(run/'results/bulk_posterior_mean_comparison.csv',plot)
    # Descriptive archive comparison only. The intervals for concordance come from the separate bootstrap.
    col='Protein_BMI_per5__posterior_mean';gate='Protein_BMI_per5__numerical_checks_pass'
    if col in linked:
        available=linked[linked.included & truth(linked[gate]) & linked[col].notna()]
        rows=[]
        for fam,g in available.groupby('family',sort=True):
            for stratum in ['all','archived_HDI_excludes_zero','archived_HDI_includes_zero']:
                gg=g if stratum=='all' else g[g.original_hdi_excludes_zero.eq(stratum.endswith('excludes_zero'))]
                if gg.empty:continue
                w=(1/gg.groupby('programme').programme.transform('size')).to_numpy()
                values=concordance_statistics(gg.posterior_mean_per_5kg,gg[col],w)
                rows.append(dict(family=fam,stratum=stratum,unique_programmes=gg.programme.nunique(),
                    feature_labels=len(gg),sign_agreement=float(values['sign_agreement'][0]),
                    correlation_of_posterior_means=float(values['correlation_of_estimated_slopes'][0]),
                    interpretation='Descriptive only; original and new Bayesian models are not jointly refitted'))
        write_csv(run/'results/archive_vs_protein_descriptive.csv',rows)
    return linked


In [ ]:
def make_figures(inputs,scores,config,run):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_pdf import PdfPages
    from matplotlib.ticker import MaxNLocator
    linked_results(scores,run)
    p=run/'results/all_effects_95HDI.csv'
    if not p.exists():return []
    all_effects=pd.read_csv(p);tab=all_effects[all_effects.variant.eq('primary')]
    figdir=run/'figures';figdir.mkdir(parents=True,exist_ok=True)
    plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans'],
        'font.size':9,'axes.titlesize':10,'axes.labelsize':9,'axes.spines.top':False,
        'axes.spines.right':False,'axes.linewidth':.7,'pdf.fonttype':42,'ps.fonttype':42,'savefig.dpi':300})
    color={'RNA':'#2C5AA0','Protein':'#D81B60','contrast':'#1F1F1F','correlation':'#6F7378'}
    names=scores['names'][:1] if config['PROFILE']=='smoke' else scores['names']
    labels={p['name']:p['label'] for p in scores['programmes']}
    counts={p['name']:len(p['matched_genes']) for p in scores['programmes']}
    title_prefix='SMOKE RUN: ' if config['PROFILE']=='smoke' else ''
    qc=tab.groupby('programme').numerical_checks_pass.apply(lambda s:truth(s).all())
    paths=[]
    def forest(panel,title):
        fig,axes=plt.subplots(1,3,figsize=(12.5,max(3.2,.46*len(panel)+1.4)),sharey=True,
            gridspec_kw={'width_ratios':[1.3,1,1]})
        for ax in axes:
            ax.axvline(0,color='#B5B8BD',lw=.8,zorder=0)
            ax.grid(axis='x',color='#E8E8EA',lw=.6);ax.set_axisbelow(True)
            ax.xaxis.set_major_locator(MaxNLocator(4))
        for y,name in enumerate(panel):
            for ax,q,dy,c in [(axes[0],'RNA_BMI_per5',-.13,color['RNA']),
                (axes[0],'Protein_BMI_per5',.13,color['Protein']),
                (axes[1],'Protein_minus_RNA_BMI_per5',0,color['contrast']),
                (axes[2],'residual_RNA_Protein_correlation',0,color['correlation'])]:
                r=tab[(tab.programme==name)&(tab.quantity==q)]
                if r.empty or not bool(qc.get(name,False)):
                    if dy>=0:ax.text(.98,y,'unfinished / check failed',transform=ax.get_yaxis_transform(),
                                     ha='right',va='center',fontsize=6,color='#A13B3B')
                    continue
                r=r.iloc[0]
                ax.plot([r.hdi_95_lower,r.hdi_95_upper],[y+dy,y+dy],color=c,lw=1.3)
                ax.plot(r.posterior_mean,y+dy,'o',color=c,ms=3.5)
        axes[0].set_yticks(range(len(panel)))
        axes[0].set_yticklabels([f'{labels[n]} ({counts[n]} genes)' for n in panel],fontsize=8)
        axes[0].set_ylim(len(panel)-.5,-.5);axes[0].set_title('BMI associations')
        axes[0].set_xlabel('Score SD per 5 kg/m²');axes[1].set_title('Protein − RNA')
        axes[1].set_xlabel('Difference in standardized associations')
        axes[2].set_title('Residual RNA–protein correlation');axes[2].set_xlabel('Correlation');axes[2].set_xlim(-1,1)
        for l in LAYERS:axes[0].plot([],[],'o',color=color[l],label=l)
        axes[0].legend(frameon=False,fontsize=8,loc='best')
        fig.suptitle(title_prefix+title,fontsize=11);fig.tight_layout();return fig
    panel=[n for n in scores['sensitivity_targets'].programme.drop_duplicates() if n in names]
    if not panel:panel=names[:1]
    fig=forest(panel,'Original signature extension: biological sensitivity panel')
    for ext in ['pdf','png']:
        path=figdir/f'original_signatures_panel.{ext}';fig.savefig(path,bbox_inches='tight');paths.append(path)
    plt.close(fig)
    fullpath=figdir/'all_original_signatures_95HDI.pdf'
    ordered=sorted(names,key=lambda n:labels[n])
    with PdfPages(fullpath) as pdf:
        for start in range(0,len(ordered),20):
            fig=forest(ordered[start:start+20],f'All eligible unique signatures: {start+1}–{min(start+20,len(ordered))}')
            pdf.savefig(fig,bbox_inches='tight');plt.close(fig)
    paths.append(fullpath)
    # All eligible programme slopes, one point per measured list. No hit-based filtering/ranking.
    wide=tab.pivot(index='programme',columns='quantity',values='posterior_mean')
    good=wide.index.intersection(qc[qc].index)
    fig,ax=plt.subplots(figsize=(5.2,4.8))
    if len(good):
        a=wide.loc[good,'RNA_BMI_per5'];b=wide.loc[good,'Protein_BMI_per5']
        ax.scatter(a,b,s=19,color=color['Protein'],alpha=.65,edgecolors='none')
        lim=max(float(np.abs(np.r_[a,b]).max())*1.15,.1)
        ax.plot([-lim,lim],[-lim,lim],color='#B5B8BD',ls='--',lw=.8)
        ax.set_xlim(-lim,lim);ax.set_ylim(-lim,lim)
    ax.axhline(0,color='#CCCCCC',lw=.7);ax.axvline(0,color='#CCCCCC',lw=.7)
    ax.set_xlabel('RNA BMI association (SD per 5 kg/m²)');ax.set_ylabel('Protein BMI association (SD per 5 kg/m²)')
    ax.set_title(title_prefix+f'All unique measured signatures ({len(good)} with numerical checks passed)')
    fig.tight_layout()
    for ext in ['pdf','png']:
        path=figdir/f'all_programme_concordance.{ext}';fig.savefig(path,bbox_inches='tight');paths.append(path)
    plt.close(fig)
    cfile=run/'bootstrap/concordance_95CI.csv'
    if cfile.exists():
        c=pd.read_csv(cfile);c=c[(c.stratum=='all') & ((c.scope=='bulk')|((c.scope=='family')&(c.comparison=='celltype_vs_Protein')))]
        categories=[('bulk','RNA → protein'),('immune_coarse','Coarse cell types → protein'),
                    ('immune_fine','Fine cell types → protein'),('nonimmune','Non-immune cell types → protein')]
        fig,axes=plt.subplots(1,2,figsize=(9,3.3),sharey=True)
        for k,(fam,label) in enumerate(categories):
            for ax,metric in zip(axes,['sign_agreement','correlation_of_estimated_slopes']):
                sub=c[(c.metric==metric)&((c.scope=='bulk') if fam=='bulk' else (c.family==fam))]
                if sub.empty:continue
                row=sub.iloc[0]
                ax.plot([row.ci_95_lower,row.ci_95_upper],[k,k],color=color['Protein'],lw=1.5)
                ax.plot(row.estimate,k,'o',color=color['Protein'],ms=4)
        axes[0].set_yticks(range(len(categories)));axes[0].set_yticklabels([x[1] for x in categories]);axes[0].invert_yaxis()
        axes[0].set_xlim(-.02,1.02);axes[1].set_xlim(-1.02,1.02)
        axes[0].set_title('Direction agreement');axes[1].set_title('Correlation of estimated slopes')
        for ax in axes:ax.set_xlabel('Linear benchmark; 95% bootstrap CI');ax.grid(axis='x',color='#E8E8EA')
        fig.suptitle(title_prefix+'Patient-resampled concordance (descriptive)',fontsize=11);fig.tight_layout()
        for ext in ['pdf','png']:
            path=figdir/f'bootstrap_concordance.{ext}';fig.savefig(path,bbox_inches='tight');paths.append(path)
        plt.close(fig)
    residual=[]
    for n in names:
        p=run/'fits/primary'/n/'patient_residuals.csv'
        if p.exists():
            d=pd.read_csv(p).set_index('base_sample_id');d.columns=[n+'__'+x for x in d.columns];residual.append(d)
    if residual:pd.concat(residual,axis=1).corr().to_csv(run/'results/between_programme_residual_correlations.csv')
    (figdir/'captions.txt').write_text(
        'Original signature panel: points are posterior means and lines are pointwise 95% HDIs. '
        'The ten named biological sensitivity targets were fixed without using bulk BMI effects; they are not the top hits. '
        'Gene counts describe matched, complete, variable RNA/protein genes. Both layers use gene z scores capped at 5, '
        'averaged and standardized in the same 127 tumour patients. Slopes are within-plex associations per 5 kg/m², '
        'adjusted for age and sex; separate plex intercepts and within/between covariates are included for both layers. '
        'Missing or numerically failed fits are labelled. Predictive and influence checks are supplied separately.\n\n'
        'All-programme scatter: one point per unique measured list, using posterior means. The diagonal is a descriptive '
        'reference; different assay reliability and SD scales prevent a mechanistic interpretation of the slope difference. '
        'The scatter has no uncertainty intervals and is not an independent replication test.\n\n'
        'Bootstrap concordance: fixed-plex linear regression benchmark, adjusted for age and sex. The same within-plex '
        'patient resamples refit original cell-type and bulk slopes, respecting original low-read exclusions and matching '
        'patients within each cell/bulk comparison. Lines are percentile bootstrap CIs, not posterior HDIs. Within each '
        'family, measured programmes receive equal total weight across their linked cell-type labels. Coarse, fine and '
        'non-immune results remain separate. These are descriptive associations, without a test against chance agreement. '
        'Scores, deconvolution and the observed plexes are held fixed. RNA reuses the original assay. Bulk protein cannot '
        'identify a cell of origin. Pointwise HDIs do not provide simultaneous error control across the panel.\n')
    return paths


In [ ]:
def final_report(inputs,scores,config,run):
    status=aggregate(run);linked_results(scores,run)
    planned=set(zip(scores['jobs'].variant,scores['jobs'].programme))
    records=status.to_dict('records')
    actual={(r['variant'],r['programme']) for r in records if r.get('postprocessing_complete') is True}
    missing=sorted(planned-actual);unresolved=[];influence_changes=[]
    for name in scores['names']:
        source=run/'fits/primary'/name/'PSIS_LOO_patients.csv'
        if not source.exists():continue
        flags=pd.read_csv(source);flags=flags[truth(flags.flag_for_refit)]
        mainfile=source.parent/'effects_95HDI.csv'
        main=pd.read_csv(mainfile).set_index('quantity') if mainfile.exists() else pd.DataFrame()
        for pid in flags.base_sample_id:
            v='primary__without_'+str(pid);s=run/'fits'/v/name/'status.json'
            st=json.loads(s.read_text()) if s.exists() else {}
            if not(st.get('postprocessing_complete') and st.get('numerical_checks_pass')):
                unresolved.append(dict(programme=name,base_sample_id=pid));continue
            f=pd.read_csv(s.parent/'effects_95HDI.csv').set_index('quantity')
            for q,row in f.iterrows():
                if q not in main.index:continue
                influence_changes.append(dict(programme=name,base_sample_id=pid,quantity=q,
                    primary_posterior_mean=main.loc[q,'posterior_mean'],deletion_posterior_mean=row.posterior_mean,
                    posterior_mean_change=row.posterior_mean-main.loc[q,'posterior_mean'],
                    deletion_hdi_95_lower=row.hdi_95_lower,deletion_hdi_95_upper=row.hdi_95_upper,
                    interpretation='Deletion refit sensitivity; not an exact LOO predictive-density correction'))
    if influence_changes:write_csv(run/'results/influence_deletion_changes.csv',influence_changes)
    complete=[r for r in records if r.get('completed') is True]
    numerical=sum(r.get('numerical_checks_pass') is True for r in complete)
    write_json(run/'results/completion_summary.json',dict(planned_primary_and_sensitivity_fits=len(planned),
        finished_planned_fits=len(planned&actual),unfinished_planned_fits=[dict(variant=v,programme=p) for v,p in missing],
        all_finished_including_influence=len(complete),numerical_checks_pass_including_influence=numerical,
        unresolved_PSIS_flags=unresolved,bootstrap_complete=(run/'bootstrap/bootstrap_complete.json').exists(),
        profile=config['PROFILE'],production_profile=config['PROFILE']=='publication',
        completion_does_not_imply_numerical_or_predictive_adequacy=True))
    print(f'Finished planned fits: {len(planned&actual)}/{len(planned)}. '
          f'Numerical checks passed: {numerical}/{len(complete)} finished fits. Unresolved PSIS flags: {len(unresolved)}.')
    if missing:print('Rerun interrupted/failed phases to resume; errors and unfinished jobs are listed in the reports.')
    if config['PROFILE']=='smoke':print('SMOKE RUN: software check only, not publication estimates.')
    (run/'results/interpretation.txt').write_text(
        'Associations with recorded BMI, per 5 kg/m². Diagnostic BMI can reflect weight loss; this analysis does not '
        'recover pre-diagnostic adiposity or a causal obesity effect. Main analyses retain the 127 original tumour patients. '
        'Original cell-type fits omit low-read patient/component profiles; paired bootstrap comparisons preserve these masks. '
        'All eligible signatures are fitted without selection on earlier hits. Duplicate measured lists are fitted once. '
        'Per-signature posterior intervals are pointwise and do not establish panel-wide discovery control. '
        'Concordance is descriptive; gene-list overlap and shared patients are retained in the joint patient bootstrap. '
        'That bootstrap refits an OLS benchmark, not the original BHM or new Student-t posteriors. Its CIs condition on '
        'fixed deconvolution, gene sets, scaling and observed plexes. Archived HDI strata are post hoc and selection-sensitive. '
        'Fraction sensitivity adds one target cell type’s RNA-derived contribution, with matched unadjusted subset; '
        'it is neither protein deconvolution nor a causal separation of abundance and state. Coherence is not technical '
        'reliability. A smaller protein association does not identify post-transcriptional regulation. Existing simulations '
        'of the cell-type model do not validate interval calibration for this extension.\n')
    return status


In [ ]:
def package_reports(run):
    dest=run/'RNA_protein_BMI_reports.zip'
    temp=dest.with_name(dest.name+'.partial')
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
        for p in sorted(run.rglob('*')):
            if not p.is_file() or p==temp or p==dest:continue
            if p.suffix=='.nc' or '.partial' in p.name:continue
            archive.write(p,p.relative_to(run))
    os.replace(temp,dest)
    print('Report ZIP:',dest)
    print('Posterior NetCDF files remain in each fit folder on Drive.')
    return dest


In [ ]:
def initialize_run(inputs, scores, payload, config, code_fingerprint):
    jobs=plan_jobs(inputs,scores,payload,config);scores['jobs']=jobs
    scientific_config={k:v for k,v in config.items() if k not in ['INPUT_ZIP','OUTPUT_BASE','CORES','FORCE_REFIT_FAILED','STOP_ON_FIT_ERROR']}
    contract=dict(analysis_version=ANALYSIS_VERSION,analysis_code_sha256=code_fingerprint,
                  input_zip_sha256=inputs['zip_sha256'],payload_sha256=canonical_hash(payload),config=scientific_config,
                  signature_sha256=payload['signature_sha256'],original_scores_sha256=payload['original_scores_sha256'],
                  genes=scores['membership'].to_dict('records'),
                  patient_ids=inputs['cohort'].base_sample_id.tolist(),scaling_reference_n=127,
                  estimand='Within-plex association per 5 kg/m2 of recorded BMI',
                  original_features=914,unique_measured_programmes=len(scores['names']),
                  independent_validation=False,missing_values_imputed=False,
                  bootstrap='Fixed-plex linear benchmark; jointly resample original scores and both bulk layers',
                  selection='All eligible original features; sensitivity targets fixed without BMI effects',
                  design_variants={v:list(VARIANTS[v]) for v in sorted(jobs.variant.unique())})
    digest=canonical_hash(contract);run=Path(config['OUTPUT_BASE'])/config['PROFILE']/digest[:12]
    run.mkdir(parents=True,exist_ok=True)
    if (run/'analysis_contract.json').exists() and json.loads((run/'analysis_contract.json').read_text())!=contract:
        raise RuntimeError('Analysis contract mismatch; refusing to mix results')
    write_json(run/'analysis_contract.json',contract)
    write_json(run/'source_payload.json',payload)
    write_csv(run/'inputs_used/patient_covariates.csv',inputs['cohort'])
    for key in ['coverage','coherence','scalers','membership','overlap','sensitivity_targets','jobs']:
        write_csv(run/f'inputs_used/{key}.csv',scores[key])
    for filename in [payload['signature_file'],'original_scores.npz']:
        target=run/'inputs_used'/filename
        if not target.exists():target.write_bytes((inputs['embedded_root']/filename).read_bytes())
    write_json(run/'inputs_used/programme_labels.json',scores['programmes'])
    target=run/'inputs_used/bulk_scores.npz'
    if not target.exists():
        np.savez_compressed(target,values=scores['Y'],programme_ids=np.asarray(scores['names'],dtype=str),
                            patient_ids=inputs['cohort'].base_sample_id.to_numpy(str),full_rna=scores['full_rna'],
                            full_rna_source_hashes=np.asarray(scores['full_rna_names'],dtype=str))
    g=inputs['cohort'].groupby('plex');bmi=inputs['cohort'].bmi
    tab=g.bmi.agg(['count','min','max','mean','std']).reset_index()
    tab=tab.merge(pd.crosstab(inputs['cohort'].plex,inputs['cohort'].bmi_group).reset_index(),on='plex')
    write_csv(run/'inputs_used/BMI_by_plex.csv',tab)
    complete=int(np.isfinite(inputs['matrices']['Protein']['values']).all(1).sum())
    write_json(run/'inputs_used/input_summary.json',dict(paired_patients=127,plexes=24,
        fully_observed_proteins=complete,modelled_original_features=914,
        unique_original_gene_lists=int(scores['coverage'].source_gene_hash.nunique()),
        eligible_original_features=int(scores['coverage'].included.sum()),
        unique_measured_programmes=len(scores['names']),planned_fits=len(jobs),
        BMI_below_18_5=int(bmi.lt(18.5).sum()),
        fraction_BMI_variation_within_plex=float(((bmi-g.bmi.transform('mean'))**2).sum()/((bmi-bmi.mean())**2).sum()),
        note='Same RNA assay as original deconvolution; protein adds a molecular layer, not cell localization.'))
    print(f'{int(scores["coverage"].included.sum())}/914 original feature labels meet coverage; '
          f'{len(scores["names"])} unique bulk scores. Planned fits: {len(jobs)} before retries/influence.')
    print('Results and checkpoints:',run)
    return run


In [ ]:
def fingerprint_analysis(function_names):
    # Ignore cell filenames and line numbers so re-executing a cell does not change its identity.
    def pack(value):
        if isinstance(value, types.CodeType):
            return dict(bytecode=value.co_code.hex(), constants=[pack(x) for x in value.co_consts],
                        names=list(value.co_names), variables=list(value.co_varnames),
                        freevars=list(value.co_freevars), cellvars=list(value.co_cellvars),
                        arguments=value.co_argcount, positional=value.co_posonlyargcount,
                        keyword_only=value.co_kwonlyargcount, flags=value.co_flags)
        if isinstance(value, bytes): return {'bytes': value.hex()}
        if isinstance(value, tuple): return [pack(x) for x in value]
        if value is None or isinstance(value, (str, int, float, bool)): return value
        return repr(value)
    return canonical_hash({name:pack(globals()[name].__code__) for name in function_names})


In [ ]:
ANALYSIS_FUNCTIONS = [
    "sha256_file",
    "canonical_hash",
    "write_json",
    "write_csv",
    "truth",
    "load_inputs",
    "build_model",
    "validate_prior",
    "posterior_array",
    "hdi",
    "effect_summary",
    "numerical_diagnostics",
    "compute_ppc",
    "compute_loo",
    "sample_model",
    "run_one",
    "aggregate",
    "package_reports",
    "load_original_context",
    "score_programmes",
    "subset_mask",
    "plan_jobs",
    "make_design",
    "initialize_run",
    "run_phase",
    "run_influence",
    "within_plex_bmi_weights",
    "concordance_statistics",
    "bootstrap_two_stage",
    "linked_results",
    "make_figures",
    "final_report"
]
CODE_FINGERPRINT = fingerprint_analysis(ANALYSIS_FUNCTIONS)

# Separate output directories prevent workers writing the same reports/checkpoints.
import shutil
ORIGINAL_OUTPUT_BASE = Path(CONFIG['OUTPUT_BASE'])
worker_base = ORIGINAL_OUTPUT_BASE / f'shards_{N_SHARDS}'
LOCAL_CONFIG = dict(CONFIG, OUTPUT_BASE=str(worker_base / (
    f'shard_{SHARD_ID:02d}' if MODE == 'worker' else 'aggregate')))
RUN_DIR = initialize_run(inputs, scores, PAYLOAD, LOCAL_CONFIG, CODE_FINGERPRINT)
ALL_JOBS = scores['jobs'].copy()
ALL_NAMES = list(scores['names'])
OWNERS = {name: i % N_SHARDS for i, name in enumerate(sorted(ALL_NAMES))}
CONTRACT = json.loads((RUN_DIR / 'analysis_contract.json').read_text())

def import_finished(source, destination, allowed_names):
    """Copy completed fits only; never modify the source run."""
    contract_file = source / 'analysis_contract.json'
    if not contract_file.exists(): return 0
    if json.loads(contract_file.read_text()) != CONTRACT:
        raise RuntimeError(f'Incompatible analysis contract: {source}')
    count = 0
    for status_path in sorted((source / 'fits').glob('*/*/status.json')):
        if status_path.parent.name not in allowed_names: continue
        state = json.loads(status_path.read_text())
        if not state.get('postprocessing_complete'): continue
        target = destination / status_path.parent.relative_to(source)
        if (target / 'status.json').exists():
            old = json.loads((target / 'status.json').read_text())
            if old.get('postprocessing_complete'): continue
        temporary = target.with_name(target.name + '.importing')
        if temporary.exists(): shutil.rmtree(temporary)
        shutil.copytree(status_path.parent, temporary)
        if target.exists(): shutil.rmtree(target)
        temporary.rename(target)
        count += 1
    return count

if MODE == 'worker':
    owned = {name for name in ALL_NAMES if OWNERS[name] == SHARD_ID}
    scores['jobs'] = ALL_JOBS[ALL_JOBS.programme.isin(owned)].copy()
    # run_influence uses names.index to index Y: filter its names with a wrapper
    # below, never change the original score-array indexing.
    if REUSE_EXISTING:
        old_run = ORIGINAL_OUTPUT_BASE / CONFIG['PROFILE'] / RUN_DIR.name
        print('Imported completed compatible fits:', import_finished(old_run, RUN_DIR, owned))
    print(f'Shard {SHARD_ID}/{N_SHARDS}: {len(owned)} programmes; {len(scores["jobs"])} planned fits')
    print(validate_prior(inputs, scores, CONFIG, RUN_DIR))
else:
    # Workers publish completion only after all their phases, including influence.
    for shard in range(N_SHARDS):
        source = worker_base / f'shard_{shard:02d}' / CONFIG['PROFILE'] / RUN_DIR.name
        marker = source / 'shard_complete.json'
        if not marker.exists():
            raise RuntimeError(f'Shard {shard} is not finished: {source}. Rerun this cell after it finishes.')
        info = json.loads(marker.read_text())
        if info.get('n_shards') != N_SHARDS or info.get('shard_id') != shard or info.get('contract_hash') != canonical_hash(CONTRACT):
            raise RuntimeError(f'Shard {shard} manifest mismatch')
        import_finished(source, RUN_DIR, {p for p in ALL_NAMES if OWNERS[p] == shard})
    print('All shards imported. No MCMC is run in aggregate mode.')
write_json(RUN_DIR / 'runtime_versions.json', {'python':sys.version,
    'packages':{p:importlib.metadata.version(p) for p in ['pymc','arviz','numpy','scipy','pandas','pytensor']}})


In [ ]:
if MODE == 'worker':
    primary_status = run_phase(inputs, scores, CONFIG, RUN_DIR, 'primary')
    display(primary_status)


In [ ]:
if MODE == 'worker':
    sensitivity_status = run_phase(inputs, scores, CONFIG, RUN_DIR, 'sensitivities')
    display(sensitivity_status)


In [ ]:
if MODE == 'worker':
    composition_status = run_phase(inputs, scores, CONFIG, RUN_DIR, 'composition')
    display(composition_status)


In [ ]:
def run_influence(inputs,scores,config,run):
    if not config['RUN_INFLUENCE_REFITS']:
        print('Deletion refits disabled; unresolved PSIS flags remain in reports.');return pd.DataFrame()
    states=[];names=scores['names'][:1] if config['PROFILE']=='smoke' else scores['names']
    for name in names:
        if MODE == 'worker' and OWNERS[name] != SHARD_ID: continue
        p=run/'fits'/'primary'/name/'PSIS_LOO_patients.csv'
        if not p.exists():continue
        flags=pd.read_csv(p);flags=flags[truth(flags.flag_for_refit)].sort_values('pareto_k',ascending=False)
        for pid in flags.base_sample_id:
            try:state=run_one(inputs,scores,config,run,scores['names'].index(name),'primary',leave_out=pid)
            except Exception as exc:
                folder=run/'fits'/('primary__without_'+str(pid))/name;folder.mkdir(parents=True,exist_ok=True)
                (folder/'error.txt').write_text(traceback.format_exc())
                state=dict(programme=name,variant='primary__without_'+str(pid),completed=False,
                           postprocessing_complete=False,numerical_checks_pass=False,error=str(exc))
                write_json(folder/'status.json',state)
                if config.get('STOP_ON_FIT_ERROR',True):raise
            states.append(state);aggregate(run)
    return pd.DataFrame(states)

if MODE == 'worker':
    influence_status = run_influence(inputs, scores, CONFIG, RUN_DIR)
    display(influence_status)
    expected = set(zip(scores['jobs'].variant, scores['jobs'].programme))
    status = aggregate(RUN_DIR)
    completed = {(r['variant'],r['programme']) for r in status.to_dict('records') if r.get('postprocessing_complete')}
    if expected - completed:
        raise RuntimeError('Some planned fits are unfinished. Resume this worker before aggregation.')
    write_json(RUN_DIR / 'shard_complete.json', dict(shard_id=SHARD_ID,
        n_shards=N_SHARDS, contract_hash=canonical_hash(CONTRACT), planned_fits=len(expected)))
    print('Shard complete. Numerical failures remain flagged in the reports:', RUN_DIR)


In [ ]:
if MODE == 'aggregate':
    concordance = bootstrap_two_stage(inputs, scores, CONFIG, RUN_DIR)
    display(concordance[(concordance.stratum == 'all') & (concordance.scope != 'celltype')])


In [ ]:
if MODE == 'aggregate':
    figure_paths = make_figures(inputs, scores, CONFIG, RUN_DIR)
    status = final_report(inputs, scores, CONFIG, RUN_DIR)
    REPORT_ZIP = package_reports(RUN_DIR)
    display(status)
    from IPython.display import Image
    for name in ['original_signatures_panel.png','all_programme_concordance.png','bootstrap_concordance.png']:
        picture = RUN_DIR / 'figures' / name
        if picture.exists(): display(Image(filename=str(picture)))
    print('Results:', RUN_DIR)
    print('Report ZIP:', REPORT_ZIP)


In [ ]:
if MODE == 'aggregate':
    # Optional: download the report. All results already remain on Drive.
    from google.colab import files
    files.download(str(REPORT_ZIP))
